In [ ]:
# /// script
# requires-python = ">=3.13"
# dependencies = [
#     "numpy",
#     "pyarrow",
# ]
# ///

In [ ]:
keys = [
    "dros.3",
    "dros.10",
    "dros.4000",
    "dros.10123",
]

In [ ]:
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

def txt_to_coord_arrays(filepath):
    x_arr = []
    y_arr = []
    z_arr = []
    with open(filepath, 'r') as file:
        for line in file:
            tokens = line.split("\t")
            x = float(tokens[0])
            y = float(tokens[1])
            z = float(tokens[2])
            x_arr.append(x)
            y_arr.append(y)
            z_arr.append(z)
    return [x_arr, y_arr, z_arr]

def arrays_to_table(x_arr, y_arr, z_arr):
    x_nparr = np.array(x_arr)
    y_nparr = np.array(y_arr)
    z_nparr = np.array(z_arr)
    return pa.Table.from_arrays([x_nparr, y_nparr, z_nparr], ["x", "y", "z"])

def write_to_file(outfile_path, table):
    with pa.OSFile(outfile_path, 'wb') as sink:
        with pa.ipc.RecordBatchFileWriter(sink, table.schema) as writer:
            writer.write_table(table)

def convert_to_arrow(filepath, outputpath):
    [x_arr, y_arr, z_arr] = txt_to_coord_arrays(filepath)
    table = arrays_to_table(x_arr, y_arr, z_arr)
    write_to_file(outfile_path, table)

def convert_to_parquet(filepath, outputpath):
    [x_arr, y_arr, z_arr] = txt_to_coord_arrays(filepath)
    table = arrays_to_table(x_arr, y_arr, z_arr)
    pq.write_table(table, outputpath, compression=None)

In [ ]:
basepath = "input"
postfix = ".txt"

out_folder = Path("out")
out_folder.mkdir(parents=True, exist_ok=True)

for key in keys:
    filepath = f"{basepath}/{key}{postfix}"
    print("input: " + filepath)

    # output: arrow
    outfile_path = "out/" + key + ".arrow"
    print("output: " + outfile_path)
    convert_to_arrow(filepath, outfile_path)
    
    # output: parquet
    outfile_path = "out/" + key + ".parquet"
    print("output: " + outfile_path)
    convert_to_parquet(filepath, outfile_path)
    print("Done.")
    print("Done.")